In [ ]:
library(tidyverse)
library(dplyr)
library(data.table)
library(ggplot2)
library(ggpubr)

In [ ]:
# related to Fig. 2d

In [ ]:
input_path<-'/data/work/pathologicalRegion/boundary_cellrate/expansion_xypos'
outpng <- paste0(input_path, '/cellrate_plot.pdf')
indf <- read.csv(paste0(input_path,"/RR_cellrate.stat.csv"))
## The input file contains the following columns: size, region, and celltypes
celltypes <- c("T","B","Plasma","Myeloid","Endothelial","Epithelial","Myofibroblast")
sumdf <- indf[,c('size','region',celltypes)]

other_region<-setdiff(unique(sort(sumdf$region)), "RR")
sumdf$size <- ifelse(sumdf$region=='RR',sumdf$size,-sumdf$size)
sumdf <- sumdf[sumdf$size >= -1500 & sumdf$size <= 1500,]

forplot <- sumdf %>% pivot_longer(cols = colnames(sumdf)[3:length(sumdf)],
                          names_to = "cell",values_to = "prop")
forplot['prop']<-forplot['prop']*100
cols <- c("#ec5e5d" ,"#377EB8","#fc98c7","#984EA3","#FC8D62","#17becf","#A65628")
names(cols) <- celltypes
len <- length(unique(forplot$size))
print(len)
p<-ggplot(forplot, aes(x = size, y = prop,color=cell)) +
  stat_summary(aes(group = cell),
    fun.data = function(y) {
      list(
        y = median(y),
        ymin = quantile(y, 0.25),
        ymax = quantile(y, 0.75)
      )
    },
    geom = "errorbar",
    width = 0.1,
    color = "grey",
    alpha = 0.6
  )+stat_summary(aes(group=cell),fun = median, geom = "point", alpha=0.8,size = 0.2) + 
  stat_summary(fun = median, geom = "line", aes(group = cell),linewidth = 0.5)+
  geom_vline(xintercept = 0, linetype = "dashed", color = "black", size = 0.5)+
  labs(title = " ", x = paste(other_region,"<-->","RR",sep=""), y = selcols)+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.text = element_text(size = 6),
    axis.line = element_line(color = "black"), 
    panel.background = element_blank(), 
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank() 
  )+
   scale_x_continuous(breaks = c(-1500,-1000,-500,0,500,1000,1500),expand = expansion(mult = c(0.01, 0.01)),limits = c(-1500, 1500))+
   scale_color_manual(aes(color=cell),values=cols)
ggsave(outpng, p, height=3, width=5.5)

In [ ]:
# related to Fig. 3e

In [ ]:
input_path<-'/data/work/pathologicalRegion/get_boundarypos/score_plot'
indf <- read.csv(paste0(input_path,"/result/fitcurve/RR.total.score.stat.txt"),sep = "\t",check.names=F)
## The input file contains the following columns: Sample, size, region, Mac_IFIT3, Mac_GPNMB, Mac_SPP1, Tumor, Mac_FOLR2
selcols <- c('Mac_IFIT3','Mac_GPNMB','Mac_SPP1','Tumor','Mac_FOLR2')
sumdf <- indf[,c('Sample','size','region',selcols)]

sumdf$Cells <- NULL
levels = unique(sort(sumdf$region))
sumdf$size <- ifelse(sumdf$region==levels[2],-sumdf$size,sumdf$size)
sumdf <- sumdf[sumdf$size >= -1500 & sumdf$size <= 1500,]

forplot=sumdf %>% pivot_longer(cols = selcols,
                          names_to = "score",values_to = "value")

forplot<-forplot[,2:5] %>% group_by(size,score) %>% mutate(value=median(value)) %>% ungroup %>% data.frame

cols <- c("#984EA3" ,"#009a3e","#c30d23","#c3c3c4",'#17becf')
names(cols) <- selcols

p <- ggplot(forplot, aes(x = size, y = value, color = factor(score))) +
  stat_smooth(
    aes(group = score, color = factor(score),fill = factor(score)), 
    method = "gam",alpha=0.2,size=0.5, level = 0.99
  ) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black", size = 0.5) +
  labs(
    x = paste(levels[2], "<-->", levels[1], sep = ""),
    y = 'Score',
    color = "Score"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.text = element_text(size = 6),
    axis.line = element_line(color = "black"), 
    panel.background = element_blank(), 
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank() 
  )+scale_color_manual(values=cols)+
  scale_fill_manual(values=cols)+
  scale_x_continuous(
    breaks = seq(-1500, 1500, by = 500), 
    limits = c(-1500, 1500)              
    )+
 guides(fill = FALSE) 
ggsave(paste0(input_path,'/result/fitcurve/plot/RR.score1500len.pdf'),p,height=1.7,width=4.5)
